# 06. Attention и RoPE: текст, видео и аудио

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** сравнить ручной attention с SDPA, измерить память и проверить геометрию вращений

**Перед началом:** K6 05; матричное умножение и softmax. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/04_attention.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §3.2](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/dit.py)
- [`core/components/rope.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/rope.py)
- [`core/tensors.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/tensors.py)
- [`runtime/kernels/sdpa.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/runtime/kernels/sdpa.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','kandinsky-6-components'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '06_attention_and_rope'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Attention — выбор источников информации

$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(QK^\top/\sqrt{d_k}+M\right)V.
$$

Q определяет, **какие tokens обновляем**. K/V определяют, **откуда берём информацию**.
Поэтому video queries + audio K/V дают обновление video, независимо от имени `va_cross_attention`.
У asymmetric widths проекции K/V переводят 896→1792 для video-query ветви и 1792→896 для audio-query.
K6 также делает RMSNorm Q/K до RoPE. Ниже проверяем само attention на уже готовых Q/K/V.

[Attention Is All You Need](https://arxiv.org/abs/1706.03762),
[RoFormer / RoPE](https://arxiv.org/abs/2104.09864) — первичные источники операций.


In [ ]:
q=torch.randn(1,2,7,8,dtype=torch.float64);k=torch.randn(1,2,5,8,dtype=torch.float64);v=torch.randn_like(k)
weights=(q@k.transpose(-1,-2)/math.sqrt(8)).softmax(-1)
manual=weights@v;sdpa=F.scaled_dot_product_attention(q,k,v)
error=(manual-sdpa).abs().max().item();assert error<1e-12
metrics['manual_sdpa_max_error']=error
fig,ax=plt.subplots(figsize=(6,3));ax.imshow(weights[0,0],aspect='auto');ax.set_xlabel('Source K/V token')
ax.set_ylabel('Updated Q token');plt.show()


## 2. Вращение сохраняет норму; позиции меняют сравнение

RoPE поворачивает пары feature coordinates, а не переставляет tokens.
Общий сдвиг позиций Q и K сохраняет их inner product в 1D случае; относительный сдвиг меняет его.
В K6 video RoPE разделяет head_dim=16+24+24=64 на T/H/W, audio использует 1D RoPE,
а `audio_freqs_scaling=0.144` задаётся checkpoint config. Позиционный scale — не sampler shift.


In [ ]:
R=native('core.components.rope');rot=native('core.tensors').apply_rotary
rope=R.RoPE1D(8,max_pos=64);q=torch.randn(4,1,8);k=torch.randn_like(q)
qr=rot(q,rope(torch.arange(4)));kr=rot(k,rope(torch.arange(4)+2))
qshift=rot(q,rope(torch.arange(4)+5));kshift=rot(k,rope(torch.arange(4)+7))
norm_error=(qr.norm(dim=-1)-q.norm(dim=-1)).abs().max().item()
shift_error=((qr*kr).sum(-1)-(qshift*kshift).sum(-1)).abs().max().item()
assert norm_error<2e-6 and shift_error<2e-6
metrics['rope_norm_error']=norm_error;metrics['rope_relative_shift_error']=shift_error
r3=R.RoPE3D((2,2,4));grid=r3((2,3,4),[torch.arange(2),torch.arange(3),torch.arange(4)])
print('3D rotary matrices shape:',grid.shape,'not a learnable embedding table')


## 3. Почему video attention дорого

Считаем логическую score matrix, не выделяя её память. 121 RGB frames при 480×864 дают
31×30×54=50220 visual tokens. Аудио — около 218 tokens. Self-attention video квадратично по Nv;
cross-modal attention пропорционально Nv×Na. FlashAttention меняет способ вычисления и хранения,
не автоматически pattern связей. Это не оценка реальной peak memory конкретного backend.


In [ ]:
Nv=31*30*54;Na=native('core.algo.prepare_latents').audio_latent_duration(31)
logical={'video self':28*Nv*Nv*2/2**30,'video queries/audio keys':28*Nv*Na*2/2**30,
         'audio queries/video keys':14*Na*Nv*2/2**30}
metrics['logical_bf16_score_GiB']=logical
print('Nv,Na:',Nv,Na,'logical BF16 score matrices GiB:',logical)
fig,ax=plt.subplots(figsize=(8,3));ax.bar(logical.keys(),logical.values());ax.set_yscale('log')
ax.set_ylabel('Logical score storage GiB, not SDPA peak');fig.tight_layout()
fig.savefig(figure_path(LAB,'attention_budget'));plt.show()


## Самостоятельное исследование

1. Покажите на shape, почему Q длиной Nv и K длиной Na возвращают Nv output tokens.
2. Удвойте H/W и посчитайте рост Nv и логической self-attention matrix.
3. Сравните audio RoPE scales 1 и .144 на одинаковых позициях; это не качество AV синхронизации.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы объясняете источник и получателя cross-attention по Q и K/V. Causal VAE, dense attention,
masking и memory-efficient attention решают разные задачи.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 05. Строительные блоки: patches, time, AdaLN и gates](05_nn_patches_and_modulation.ipynb) · [07. От DiT к CrossDiT: как denoiser устроен изнутри →](07_dit_and_text_crossdit.ipynb)
